<a href="https://colab.research.google.com/github/Anshikaag-28/Anshika-FlyRank-ML/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Anshikaag-28/Anshika-FlyRank-ML/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*


The validated Random Forest ranking is used as a **prioritization tool** for content review. The goal is not to automatically declare that a page needs a refresh, but to identify pages that should be reviewed first.

The primary action is:

**Action: REVIEW_FOR_REFRESH**

The main reason code is:

**Reason code: DECLINING_CONTENT_OPPORTUNITY**

This reason code means that the page is associated with the observed declining proxy used in the model and has been ranked as a higher-priority candidate for human review.

The ranking should be interpreted as:

> **Review these pages first because the model found stronger evidence of the observed declining pattern.**

It should not be interpreted as:

> **These pages are guaranteed to need a refresh.**

### Archetype → action mapping

| Archetype                           | Main signal                                      | Suggested action   |
| ----------------------------------- | ------------------------------------------------ | ------------------ |
| High-priority declining opportunity | High model score and observed decline proxy      | REVIEW_FOR_REFRESH |
| Moderate opportunity                | Moderate model score with useful search exposure | REVIEW_FOR_REFRESH |
| Low-confidence / low-signal page    | Low model score or weak evidence                 | MONITOR            |
| Insufficient evidence               | Very low exposure or incomplete signals          | HOLD               |

The action is therefore a recommendation for **human review**, rather than an automatic content change.

The model's measured Precision@50 under the client-grouped validation was **0.84**. This means that 42 of the top 50 ranked test examples were positive according to the observed proxy, on this particular evaluated split. It does not mean that 84% of all pages need refreshing.

The decay/refresh insight should also be treated directionally. The data can help identify pages that deserve review, but it does not establish that refreshing a page will cause recovery.


In [9]:
# W07 Section 1
# Recreate the validated W06 model output and build the ranked action queue

import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

# -----------------------------
# 1. Load data
# -----------------------------
repo_path = Path("/content/Anshika-FlyRank-ML")

if not repo_path.exists():
    !git clone -q https://github.com/Anshikaag-28/Anshika-FlyRank-ML.git /content/Anshika-FlyRank-ML

%cd /content/Anshika-FlyRank-ML

data_path = Path("data/raw/content_refresh_anonymized.csv")
df = pd.read_csv(data_path)

# -----------------------------
# 2. Create proxy target
# -----------------------------
df["is_declining_label"] = (
    df["trend_direction"]
    .astype(str)
    .str.lower()
    .eq("down")
    .astype(int)
)

# -----------------------------
# 3. Use the validated W06 features
# -----------------------------
features = [
    "impressions_90d",
    "sessions_90d",
    "content_age_days",
    "ctr",
    "avg_position",
    "word_count",
    "engagement_rate"
]

# -----------------------------
# 4. Client-grouped split
# -----------------------------
model_df = df.dropna(
    subset=features + ["is_declining_label", "client_id"]
).copy()

clients = model_df["client_id"].unique()

train_clients, test_clients = train_test_split(
    clients,
    test_size=0.20,
    random_state=42
)

train_df = model_df[
    model_df["client_id"].isin(train_clients)
].copy()

test_df = model_df[
    model_df["client_id"].isin(test_clients)
].copy()

# Check no client leakage
overlap = set(train_df["client_id"]).intersection(
    set(test_df["client_id"])
)

assert len(overlap) == 0

# -----------------------------
# 5. Prepare model data
# -----------------------------
X_train = train_df[features].copy()
y_train = train_df["is_declining_label"].copy()

X_test = test_df[features].copy()
y_test = test_df["is_declining_label"].copy()

train_medians = X_train.median(numeric_only=True)

X_train = X_train.fillna(train_medians)
X_test = X_test.fillna(train_medians)

# -----------------------------
# 6. Train validated Random Forest
# -----------------------------
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

rf_model.fit(X_train, y_train)

# -----------------------------
# 7. Generate ranking scores
# -----------------------------
model_scores = rf_model.predict_proba(X_test)[:, 1]

# -----------------------------
# 8. Build ranked queue
# -----------------------------
action_queue = test_df[
    ["content_id", "client_id"] + features
].copy()

action_queue["model_score"] = model_scores

action_queue = action_queue.sort_values(
    "model_score",
    ascending=False
).reset_index(drop=True)

action_queue["rank"] = np.arange(
    1,
    len(action_queue) + 1
)

# -----------------------------
# 9. Define action bands
# -----------------------------
n = len(action_queue)

action_queue["archetype"] = np.select(
    [
        action_queue["rank"] <= 50,
        action_queue["rank"] <= int(n * 0.20),
        action_queue["rank"] <= int(n * 0.50)
    ],
    [
        "HIGH_PRIORITY_OPPORTUNITY",
        "MODERATE_OPPORTUNITY",
        "LOW_PRIORITY_SIGNAL"
    ],
    default="HOLD"
)

action_queue["action"] = np.select(
    [
        action_queue["archetype"] == "HIGH_PRIORITY_OPPORTUNITY",
        action_queue["archetype"] == "MODERATE_OPPORTUNITY",
        action_queue["archetype"] == "LOW_PRIORITY_SIGNAL"
    ],
    [
        "REVIEW_FOR_REFRESH",
        "REVIEW_FOR_REFRESH",
        "MONITOR"
    ],
    default="HOLD"
)

action_queue["reason_code"] = np.select(
    [
        action_queue["archetype"] == "HIGH_PRIORITY_OPPORTUNITY",
        action_queue["archetype"] == "MODERATE_OPPORTUNITY",
        action_queue["archetype"] == "LOW_PRIORITY_SIGNAL"
    ],
    [
        "HIGH_MODEL_RANK",
        "MODERATE_MODEL_RANK",
        "LOW_MODEL_RANK"
    ],
    default="LOW_PRIORITY_SIGNAL"
)

# -----------------------------
# 10. Arrange columns
# -----------------------------
action_queue = action_queue[
    [
        "rank",
        "content_id",
        "client_id",
        "model_score",
        "archetype",
        "reason_code",
        "action"
    ] + features
]

print("Validated test rows:", len(action_queue))
print("Test clients:", len(test_clients))
print("Client overlap:", len(overlap))
print("\nTop 10 ranked actions:")

display(action_queue.head(10))

/content/Anshika-FlyRank-ML
Validated test rows: 2812
Test clients: 7
Client overlap: 0

Top 10 ranked actions:


,rank,content_id,client_id,model_score,archetype,reason_code,action,impressions_90d,sessions_90d,content_age_days,ctr,avg_position,word_count,engagement_rate
0,1,content_569e0d485db8,client_349c41201b,0.812022,HIGH_PRIORITY_OPPORTUNITY,HIGH_MODEL_RANK,REVIEW_FOR_REFRESH,1816,10,154,0.06,28.1,4401.0,0.00
1,2,content_8d387d58f71f,client_349c41201b,0.810363,HIGH_PRIORITY_OPPORTUNITY,HIGH_MODEL_RANK,REVIEW_FOR_REFRESH,1032,16,148,0.00,5.7,6583.0,0.00
2,3,content_a8cd736c9c7b,client_349c41201b,0.809200,HIGH_PRIORITY_OPPORTUNITY,HIGH_MODEL_RANK,REVIEW_FOR_REFRESH,2877,4,144,0.03,23.2,4448.0,0.00
3,4,content_c14a055e6a26,client_349c41201b,0.808771,HIGH_PRIORITY_OPPORTUNITY,HIGH_MODEL_RANK,REVIEW_FOR_REFRESH,8605,30,140,0.07,34.0,4148.0,0.00
4,5,content_be1c109fcdbc,client_349c41201b,0.807584,HIGH_PRIORITY_OPPORTUNITY,HIGH_MODEL_RANK,REVIEW_FOR_REFRESH,10931,19,144,0.05,34.0,5359.0,0.00
5,6,content_cdc1d5c2b05a,client_19581e27de,0.806394,HIGH_PRIORITY_OPPORTUNITY,HIGH_MODEL_RANK,REVIEW_FOR_REFRESH,3989,40,153,0.00,17.0,2910.0,2.50
6,7,content_0bc3bda5785c,client_349c41201b,0.805916,HIGH_PRIORITY_OPPORTUNITY,HIGH_MODEL_RANK,REVIEW_FOR_REFRESH,2929,5,167,0.07,16.9,3420.0,0.00
7,8,content_a451a30f3920,client_19581e27de,0.803025,HIGH_PRIORITY_OPPORTUNITY,HIGH_MODEL_RANK,REVIEW_FOR_REFRESH,11926,13,139,0.04,4.4,4028.0,7.69
8,9,content_39dbf0817270,client_19581e27de,0.802735,HIGH_PRIORITY_OPPORTUNITY,HIGH_MODEL_RANK,REVIEW_FOR_REFRESH,3755,2,98,0.05,1.3,2533.0,0.00
9,10,content_7ee674751acf,client_349c41201b,0.802707,HIGH_PRIORITY_OPPORTUNITY,HIGH_MODEL_RANK,REVIEW_FOR_REFRESH,526,13,168,0.00,21.3,4053.0,0.00


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

In [10]:
# W07 Section 2
# Intended use, limits, and cost/value summary

top_50 = action_queue.head(50)

proxy_positive_count = int(
    test_df.iloc[
        action_queue.head(50).index
    ]["is_declining_label"].sum()
)

precision_at_50 = proxy_positive_count / 50

print("Top 50 pages:", len(top_50))
print("Positive proxy pages in Top 50:", proxy_positive_count)
print("Measured Precision@50:", round(precision_at_50, 3))

print("\nAction distribution:")
print(action_queue["action"].value_counts())

print("\nArchetype distribution:")
print(action_queue["archetype"].value_counts())

print("\nReason-code distribution:")
print(action_queue["reason_code"].value_counts())

print("\nHighest model score:",
      round(action_queue["model_score"].max(), 4))

print("Lowest model score:",
      round(action_queue["model_score"].min(), 4))

print("\nIntended use:")
print("Use the ranking to help human reviewers decide which pages to inspect first.")

print("\nCost/value principle:")
print("Review capacity is limited, so higher-ranked pages can be reviewed first.")
print("This is a prioritization principle, not a measured monetary return.")

print("\nLimits:")
print("The target is an observed declining proxy, not guaranteed future decline.")
print("The model supports human review; it does not automatically decide to refresh content.")
print("The measured Precision@50 does not prove that refreshing a page will improve performance.")

Top 50 pages: 50
Positive proxy pages in Top 50: 26
Measured Precision@50: 0.52

Action distribution:
action
HOLD                  1406
MONITOR                844
REVIEW_FOR_REFRESH     562
Name: count, dtype: int64

Archetype distribution:
archetype
HOLD                         1406
LOW_PRIORITY_SIGNAL           844
MODERATE_OPPORTUNITY          512
HIGH_PRIORITY_OPPORTUNITY      50
Name: count, dtype: int64

Reason-code distribution:
reason_code
LOW_PRIORITY_SIGNAL    1406
LOW_MODEL_RANK          844
MODERATE_MODEL_RANK     512
HIGH_MODEL_RANK          50
Name: count, dtype: int64

Highest model score: 0.812
Lowest model score: 0.001

Intended use:
Use the ranking to help human reviewers decide which pages to inspect first.

Cost/value principle:
Review capacity is limited, so higher-ranked pages can be reviewed first.
This is a prioritization principle, not a measured monetary return.

Limits:
The target is an observed declining proxy, not guaranteed future decline.
The model suppor

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

In [11]:
# W07 Section 3
# Human review checklist and automation boundaries

review_queue = action_queue.head(50).copy()

print("HUMAN REVIEW CHECKLIST")
print("1. Check whether the page has meaningful search exposure.")
print("2. Check whether the observed decline is large enough to investigate.")
print("3. Check for seasonality or temporary changes.")
print("4. Check whether related pages overlap or compete.")
print("5. Check whether search intent or SERP conditions changed.")
print("6. Check whether the content is still relevant to the user intent.")
print("7. Check whether important information is outdated or incomplete.")
print("8. Decide whether a realistic content improvement is possible.")

print("\nNO-GO LIST")
print("1. Do not automatically publish rewritten content.")
print("2. Do not automatically delete or consolidate pages.")
print("3. Do not automatically change search intent or content strategy.")
print("4. Do not automatically rewrite titles or content from the score alone.")
print("5. Do not claim that a refresh will recover traffic.")
print("6. Do not use the model score as proof of causality.")
print("7. Do not make business-critical decisions from the score alone.")

print("\nTOP REVIEW QUEUE")
display(
    review_queue[
        [
            "rank",
            "content_id",
            "model_score",
            "archetype",
            "reason_code",
            "action",
            "impressions_90d",
            "sessions_90d",
            "content_age_days",
            "ctr",
            "avg_position"
        ]
    ]
)

HUMAN REVIEW CHECKLIST
1. Check whether the page has meaningful search exposure.
2. Check whether the observed decline is large enough to investigate.
3. Check for seasonality or temporary changes.
4. Check whether related pages overlap or compete.
5. Check whether search intent or SERP conditions changed.
6. Check whether the content is still relevant to the user intent.
7. Check whether important information is outdated or incomplete.
8. Decide whether a realistic content improvement is possible.

NO-GO LIST
1. Do not automatically publish rewritten content.
2. Do not automatically delete or consolidate pages.
3. Do not automatically change search intent or content strategy.
4. Do not automatically rewrite titles or content from the score alone.
5. Do not claim that a refresh will recover traffic.
6. Do not use the model score as proof of causality.
7. Do not make business-critical decisions from the score alone.

TOP REVIEW QUEUE


,rank,content_id,model_score,archetype,reason_code,action,impressions_90d,sessions_90d,content_age_days,ctr,avg_position
0,1,content_569e0d485db8,0.812022,HIGH_PRIORITY_OPPORTUNITY,HIGH_MODEL_RANK,REVIEW_FOR_REFRESH,1816,10,154,0.06,28.1
1,2,content_8d387d58f71f,0.810363,HIGH_PRIORITY_OPPORTUNITY,HIGH_MODEL_RANK,REVIEW_FOR_REFRESH,1032,16,148,0.00,5.7
2,3,content_a8cd736c9c7b,0.809200,HIGH_PRIORITY_OPPORTUNITY,HIGH_MODEL_RANK,REVIEW_FOR_REFRESH,2877,4,144,0.03,23.2
3,4,content_c14a055e6a26,0.808771,HIGH_PRIORITY_OPPORTUNITY,HIGH_MODEL_RANK,REVIEW_FOR_REFRESH,8605,30,140,0.07,34.0
4,5,content_be1c109fcdbc,0.807584,HIGH_PRIORITY_OPPORTUNITY,HIGH_MODEL_RANK,REVIEW_FOR_REFRESH,10931,19,144,0.05,34.0
5,6,content_cdc1d5c2b05a,0.806394,HIGH_PRIORITY_OPPORTUNITY,HIGH_MODEL_RANK,REVIEW_FOR_REFRESH,3989,40,153,0.00,17.0
6,7,content_0bc3bda5785c,0.805916,HIGH_PRIORITY_OPPORTUNITY,HIGH_MODEL_RANK,REVIEW_FOR_REFRESH,2929,5,167,0.07,16.9
7,8,content_a451a30f3920,0.803025,HIGH_PRIORITY_OPPORTUNITY,HIGH_MODEL_RANK,REVIEW_FOR_REFRESH,11926,13,139,0.04,4.4
8,9,content_39dbf0817270,0.802735,HIGH_PRIORITY_OPPORTUNITY,HIGH_MODEL_RANK,REVIEW_FOR_REFRESH,3755,2,98,0.05,1.3
9,10,content_7ee674751acf,0.802707,HIGH_PRIORITY_OPPORTUNITY,HIGH_MODEL_RANK,REVIEW_FOR_REFRESH,526,13,168,0.00,21.3


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

In [12]:
# W07 Section 4
# Monitoring baseline and retrain/review triggers

monitoring_summary = {
    "validated_precision_at_50": round(float(precision_at_50), 3),
    "queue_rows": len(action_queue),
    "top_50_proxy_positive_count": proxy_positive_count,
    "mean_model_score": float(action_queue["model_score"].mean()),
    "median_model_score": float(action_queue["model_score"].median()),
    "high_priority_rows": int(
        (action_queue["archetype"] == "HIGH_PRIORITY_OPPORTUNITY").sum()
    ),
    "moderate_priority_rows": int(
        (action_queue["archetype"] == "MODERATE_OPPORTUNITY").sum()
    ),
    "monitor_rows": int(
        (action_queue["action"] == "MONITOR").sum()
    )
}

monitoring_df = pd.DataFrame([monitoring_summary])

display(monitoring_df)

print("\nMONITORING SIGNALS")
print("1. Precision@50 on newly evaluated data.")
print("2. Distribution of model scores.")
print("3. Distribution of important input features.")
print("4. Number of high-priority recommendations.")
print("5. Human acceptance or rejection of recommendations.")
print("6. Changes in the target/proxy definition.")
print("7. Changes in data collection or feature definitions.")

print("\nREVIEW / RETRAIN TRIGGERS")
print("1. Precision@50 falls substantially from the validated baseline.")
print("2. Input feature distributions change substantially.")
print("3. The size of the high-priority queue changes unexpectedly.")
print("4. Human reviewers frequently reject recommendations.")
print("5. The target or proxy definition changes.")
print("6. The underlying population of pages or clients changes.")

,validated_precision_at_50,queue_rows,top_50_proxy_positive_count,mean_model_score,median_model_score,high_priority_rows,moderate_priority_rows,monitor_rows
0,0.52,2812,26,0.526368,0.557171,50,512,844



MONITORING SIGNALS
1. Precision@50 on newly evaluated data.
2. Distribution of model scores.
3. Distribution of important input features.
4. Number of high-priority recommendations.
5. Human acceptance or rejection of recommendations.
6. Changes in the target/proxy definition.
7. Changes in data collection or feature definitions.

REVIEW / RETRAIN TRIGGERS
1. Precision@50 falls substantially from the validated baseline.
2. Input feature distributions change substantially.
3. The size of the high-priority queue changes unexpectedly.
4. Human reviewers frequently reject recommendations.
5. The target or proxy definition changes.
6. The underlying population of pages or clients changes.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [13]:
# W07 Section 5
# Export the ranked queue and monitoring metrics

output_dir = Path("work/outputs")
output_dir.mkdir(parents=True, exist_ok=True)

figure_dir = Path("work/figures")
figure_dir.mkdir(parents=True, exist_ok=True)

# Export ranked queue
queue_path = output_dir / "action_playbook_queue.csv"

action_queue.to_csv(
    queue_path,
    index=False
)

# Export monitoring metrics
metrics_path = output_dir / "action_playbook_metrics.json"

monitoring_df.to_json(
    metrics_path,
    orient="records",
    indent=2
)

# Verify files
assert queue_path.exists()
assert metrics_path.exists()

check_queue = pd.read_csv(queue_path)

print("Export completed.")
print("Queue file:", queue_path)
print("Metrics file:", metrics_path)
print("Rows exported:", len(check_queue))

print("\nExported queue preview:")
display(check_queue.head(10))

FileExistsError: [Errno 17] File exists: 'work/outputs'

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.